In [1]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import optuna
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

name = "gdsc1"
# task = "cell"
task = "drug"


method = "GCN"
PATH = f"../{name}_data/"

target_dim = [
    1,  # Drug
    # 0  # Cell
]

In [2]:
from get_params import get_params
from metrics import compute_metrics_stats

from drGAT import No_atten_drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler

In [23]:
(
    res,
    pos_num,
    null_mask,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

load gdsc1
unique drugs: 73
unique genes: 166
DTI unique genes:  166
Top 90% variable genes:  1957
Total:  2099
Final gene exp shape: (916, 2099)
Final drug Act shape: (331, 916)


100%|██████████| 25/25 [00:01<00:00, 14.94it/s]


Done!


In [24]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    params,
    device,
    seed,
):
    sampler = NewSampler(
        res_mat,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )

    (_, best_val_labels, best_val_prob, best_metrics, _, _, _) = No_atten_drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    # print(best_val_labels)
    return best_val_labels, best_val_prob

In [25]:
params = {
    "n_drug": S_d.shape[0],
    "n_cell": S_c.shape[0],
    "n_gene": S_g.shape[0],
    "dropout1": 0.1,
    "dropout2": 0.1,
    "hidden1": 256,
    "hidden2": 128,
    "hidden3": 10,
    "epochs": 1,
    # trial.suggest_int("epochs", 100, 10000, step=100),
    "heads": 1,
    "activation": "relu",
    "optimizer": "Adam",
    "lr": 0.1,
    "weight_decay": 1e-2,
    "scheduler": None,
    "gnn_layer": method,
}

In [26]:
# res = res.iloc[:, :100]

In [27]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        p = res.iloc[:, target_index].dropna() > 0
        tmp = sum(p) * 100 / len(p)
        if 0 < tmp < 100:
            if dim:
                if drug_sum[target_index] < 10:
                    continue
            else:
                if cell_sum[target_index] < 10:
                    continue

            for fold in range(n_kfold):
                true_data, predict_data = drGAT_new(
                    res_mat=res,
                    null_mask=null_mask.T.values,
                    target_dim=dim,
                    target_index=target_index,
                    S_d=S_d,
                    S_c=S_c,
                    S_g=S_g,
                    A_cg=A_cg,
                    A_dg=A_dg,
                    PATH=PATH,
                    params=params,
                    device=device,
                    seed=seed,
                )
            true_datas = pd.concat(
                [true_datas, pd.DataFrame(true_data).T], ignore_index=True
            )
            predict_datas = pd.concat(
                [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
            )
        else:
            print("ttt")

0it [00:00, ?it/s]

Using device: cuda



100%|██████████| 1/1 [00:02<00:00,  2.92s/it]
1it [00:07,  7.58s/it]

Best model found at epoch 1
[0. 1. 1. 0. 0. 0. 1. 1. 0. 0. 1. 0. 0. 0. 1. 1. 0. 1. 1. 1. 0. 1. 0. 1.
 1. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 1. 1. 0. 1. 1.
 1. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0. 1. 1. 0. 1. 0. 1. 0. 1. 1.
 1. 0. 1. 0. 1. 0. 0. 0. 1. 1. 0. 0. 1. 0. 0. 1. 1. 0. 1. 1. 1. 0. 1. 1.
 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0.
 1. 1. 1. 1. 0. 0. 1. 0. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1.
 1. 0. 0. 0. 0. 1. 0. 1. 1. 0. 1. 1. 0. 0. 0. 1. 0. 1. 1. 1. 1. 1. 0. 0.
 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 1. 0. 0. 1. 1. 0. 1. 0. 0.
 1. 0. 1. 1. 1. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 0.
 1. 1. 1. 1. 1. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 1. 1. 0. 1. 0. 0. 0. 0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 1. 1.
 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 0.
 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0. 1. 1. 0. 0. 1. 1. 0. 0.
 1. 1. 1. 1. 1. 1. 1. 1


100%|██████████| 1/1 [00:00<00:00, 20.89it/s]
2it [00:11,  5.59s/it]

...

Best model found at epoch 1
[1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1.
 1. 1. 0. 0. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 0. 0. 0. 1. 1. 1. 1. 1. 1. 0.
 1. 0. 1. 0. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0. 1.
 0. 1. 1. 0. 1. 0. 1. 1. 1. 1. 0. 0. 0. 1. 0. 0. 1. 1. 1. 1. 0. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1.
 0. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1.
 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 1.
 0. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 0.
 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 1. 1. 1. 0. 1.
 1. 0. 1. 0. 1. 0. 1. 1. 0. 0. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0.
 0. 0. 0. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1.
 1. 1. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 1.
 1. 1. 1. 1. 0. 0. 1. 0. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 0.
 1. 1. 0. 1. 0. 1. 0. 1


100%|██████████| 1/1 [00:00<00:00, 20.93it/s]
4it [00:20,  4.69s/it]

Best model found at epoch 1
[1. 0. 1. 1. 1. 0. 0. 1. 1. 1. 0. 1. 0. 0. 1. 1. 1. 0. 0. 1. 1. 0. 1. 0.
 0. 1. 0. 1. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 1. 0. 0. 1. 1. 0. 1. 1. 1. 1.
 1. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0.
 1. 1. 1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 1. 0. 1. 1. 1. 1.
 0. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 0.]
ttt


5it [00:24,  4.86s/it]


KeyboardInterrupt: 

In [28]:
true_data

array([1., 0., 1., 1., 1., 0., 0., 1., 1., 1., 0., 1., 0., 0., 1., 1., 1.,
       0., 0., 1., 1., 0., 1., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0., 1.,
       1., 0., 1., 0., 1., 0., 0., 1., 1., 0., 1., 1., 1., 1., 1., 0., 0.,
       0., 0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0.,
       0., 0., 0., 0., 1., 1., 1., 0., 0., 1., 1., 1., 0., 0., 1., 0., 0.,
       1., 0., 0., 0., 0., 1., 0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1.,
       1., 0., 0., 1., 0., 0.], dtype=float32)

In [29]:
metrics_result = compute_metrics_stats(
    true=true_datas,
    pred=predict_datas,
    target_metrics=["AUROC", "AUPR", "F1", "ACC"],
)

In [30]:
metrics_result

{'raw': {'means': {'ACC': 0.4383245382585752,
   'Precision': 0.0,
   'Recall': 0.0,
   'F1': 0.0,
   'AUROC': 0.5325694566668783,
   'AUPR': 0.5967251111931888,
   'MCC': 0.0,
   'Specificity': 1.0,
   'Balanced_ACC': 0.5,
   'LogLoss': 0.7905378284198348,
   'Brier': 0.29394316818646676},
  'stds': {'ACC': 0.1233509234828496,
   'Precision': 0.0,
   'Recall': 0.0,
   'F1': 0.0,
   'AUROC': 0.05513251911210641,
   'AUPR': 0.11784399242142088,
   'MCC': 0.0,
   'Specificity': 0.0,
   'Balanced_ACC': 0.0,
   'LogLoss': 0.08543160709190038,
   'Brier': 0.03773361764029053}},
 'formatted': {'ACC': '0.438 (±0.123)',
  'Precision': '0.000 (±0.000)',
  'Recall': '0.000 (±0.000)',
  'F1': '0.000 (±0.000)',
  'AUROC': '0.533 (±0.055)',
  'AUPR': '0.597 (±0.118)',
  'MCC': '0.000 (±0.000)',
  'Specificity': '1.000 (±0.000)',
  'Balanced_ACC': '0.500 (±0.000)',
  'LogLoss': '0.791 (±0.085)',
  'Brier': '0.294 (±0.038)'},
 'target_values': [0.5325694566668783,
  0.5967251111931888,
  0.0,
  0.438